## Data Acquisition and Cleaning

This notebook retrieves the official NSW EV charging data and ABS SA4 boundaries, inspects data quality, and prepares the charger dataset for spatial integration and augmentation.

In [6]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download the EV Charging Dataset

Retrieve `ev_20251216.csv` from Transport for NSW and save the original
file in `data/raw/`.


In [7]:
from urllib.request import Request, urlopen
import csv
import io

EV_URL = (
    "https://opendata.transport.nsw.gov.au/data/dataset/"
    "be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/"
    "7bbb6461-e52d-4fe7-ace4-a15c30198de0/"
    "download/ev_20251216.csv"
)

request = Request(
    EV_URL,
    headers={"User-Agent": "COMP5339-Student-Project/1.0"},
)

with urlopen(request, timeout=60) as response:
    content = response.read()

text = content.decode("utf-8-sig")
reader = csv.DictReader(io.StringIO(text))
required_columns = {"Station_address", "Charger_Type", "Latitude", "Longitude"}

if not required_columns.issubset(reader.fieldnames or []):
    raise ValueError("The response is not the expected charger CSV.")

rows = list(reader)

output_file = RAW_DIR / "ev_20251216.csv"
output_file.write_bytes(content)

print("Download completed:", output_file.name)
print("Rows:", len(rows))
print("Columns:", len(reader.fieldnames))

Download completed: ev_20251216.csv
Rows: 1958
Columns: 12


## 2. Download the SA4 Boundary Dataset

Download and extract the ABS ASGS Edition 4 SA4 boundaries (2026, GDA2020). Preserve the original ZIP and all shapefile components.

In [8]:
from urllib.request import Request, urlopen
from zipfile import ZipFile
from io import BytesIO

SA4_URL = (
    "https://www.abs.gov.au/statistics/standards/"
    "australian-statistical-geography-standard-asgs/"
    "edition-4-july-2026-june-2031/access-and-downloads/"
    "digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip"
)

request = Request(
    SA4_URL,
    headers={"User-Agent": "COMP5339-Student-Project/1.0"},
)

with urlopen(request, timeout=120) as response:
    zip_content = response.read()

zip_path = RAW_DIR / "SA4_2026_AUST_SHP_GDA2020.zip"
extract_dir = RAW_DIR / "sa4_2026"

with ZipFile(BytesIO(zip_content)) as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise ValueError(f"Corrupted archive member: {bad_file}")

    extraction_root = extract_dir.resolve()
    for member in archive.infolist():
        target = (extract_dir / member.filename).resolve()
        if not target.is_relative_to(extraction_root):
            raise ValueError("The archive contains an unsafe path.")

    zip_path.write_bytes(zip_content)
    extract_dir.mkdir(parents=True, exist_ok=True)
    archive.extractall(extract_dir)

shapefiles = list(extract_dir.rglob("*.shp"))
if not shapefiles:
    raise ValueError("No shapefile was found in the archive.")

print("Download completed:", zip_path.name)
print("Shapefiles found:", len(shapefiles))

for shapefile in shapefiles:
    for extension in (".shp", ".shx", ".dbf", ".prj"):
        if not shapefile.with_suffix(extension).exists():
            raise FileNotFoundError(
                f"Missing shapefile component: {extension}"
            )
    print("Shapefile components verified:", shapefile.name)

Download completed: SA4_2026_AUST_SHP_GDA2020.zip
Shapefiles found: 1
Shapefile components verified: SA4_2026_AUST_GDA2020.shp


## 3. Inspect the Raw EV Charging Data

3.1 
Before applying any data cleaning steps, the dataset will first be inspected to understand its structure and data quality. The initial analysis will:
1. Check the total number of rows and identify all available columns.
2. Examine missing values and determine which columns contain null or empty entries.
3. Check for fully duplicated rows where all column values are identical.
4. Review the distribution and counts of AC/DC types and source categories.
5. Use the findings from these checks to determine appropriate data-cleaning rules.


In [9]:
import csv
from collections import Counter

ev_file = RAW_DIR / "ev_20251216.csv"

with ev_file.open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    columns = reader.fieldnames
    records = list(reader)

if not columns or not records:
    raise ValueError("The CSV file has no columns or data records.")

print("ROW COUNT:", len(records))
print("COLUMN COUNT:", len(columns))

print("\nCOLUMNS:")
for column in columns:
    print(column)

print("\nFIRST THREE RECORDS:")
for record in records[:3]:
    print(record)

print("\nBLANK VALUES BY COLUMN:")
for column in columns:
    blank_count = sum(
        record[column] is None or record[column].strip() == ""
        for record in records
    )
    print(f"{column}: {blank_count}")

row_values = [
    tuple(record[column] for column in columns)
    for record in records
]
duplicate_count = len(row_values) - len(set(row_values))
print("\nEXACT DUPLICATE ROWS:", duplicate_count)

print("\nCHARGER TYPE COUNTS:")
type_counts = Counter(record["Charger_Type"] for record in records)
for charger_type, count in type_counts.items():
    print(f"{charger_type!r}: {count}")

print("\nSOURCE COUNTS:")
source_counts = Counter(record["Source"] for record in records)
for source, count in source_counts.items():
    print(f"{source!r}: {count}")

ROW COUNT: 1958
COLUMN COUNT: 12

COLUMNS:
OBJECTID
Station_name
Station_address
Operator
Number_of_plugs
Charger_Type
Charger_rating
Latitude
Longitude
LGANAME
PCODE
Source

FIRST THREE RECORDS:
{'OBJECTID': '', 'Station_name': '', 'Station_address': ', Muswellbrook, 2333', 'Operator': 'EVUp', 'Number_of_plugs': '2', 'Charger_Type': 'AC', 'Charger_rating': '22 kW', 'Latitude': '-32.26224229', 'Longitude': '150.8901391', 'LGANAME': 'Muswellbrook Shire Council', 'PCODE': '2333', 'Source': 'Existing Destination Chargers'}
{'OBJECTID': '', 'Station_name': '', 'Station_address': '01 Wallgrove Road, Sydney, 2766', 'Operator': 'BP', 'Number_of_plugs': '4', 'Charger_Type': 'DC', 'Charger_rating': '150 kW', 'Latitude': '-33.81100405', 'Longitude': '150.8495966', 'LGANAME': 'Blacktown City Council', 'PCODE': '2766', 'Source': 'Existing Fast Chargers'}
{'OBJECTID': '', 'Station_name': '', 'Station_address': '1 - 7 Ross St, Wilcannia NSW 2836, Australia', 'Operator': 'NRMA', 'Number_of_plugs': '4

### 3.2 Numeric and Categorical Data Checks

The next step checks whether latitude, longitude, and plug-count values can be converted to finite numbers and examines their ranges for potential anomalies. Operator names and charging-power values are reviewed for differences in spelling, formatting, and units. Records with missing values in LGANAME, PCODE, and Source are also checked to determine whether all three fields are missing in the same rows. These checks inform subsequent cleaning decisions without modifying the original data.

In [10]:
import math
from collections import Counter

numeric_columns = [
    "Latitude",
    "Longitude",
    "Number_of_plugs",
]

for column in numeric_columns:
    valid_values = []
    invalid_values = []

    for csv_row, record in enumerate(records, start=2):
        raw_value = record[column]

        try:
            value = float(raw_value)
            if not math.isfinite(value):
                raise ValueError("Non-finite value")
            valid_values.append(value)
        except (TypeError, ValueError):
            invalid_values.append((csv_row, raw_value))

    print(f"\nNUMERIC CHECK: {column}")
    print("Invalid values:", len(invalid_values))
    print("Invalid examples:", invalid_values[:10])

    if valid_values:
        print("Minimum:", min(valid_values))
        print("Maximum:", max(valid_values))

for column in ["Operator", "Charger_rating"]:
    print(f"\nVALUE COUNTS: {column}")
    counts = Counter(record[column] for record in records)

    for value, count in counts.most_common():
        print(f"{value!r}: {count}")

missing_region_fields = ["LGANAME", "PCODE", "Source"]

all_three_blank = sum(
    all(not record[column].strip() for column in missing_region_fields)
    for record in records
)

print("\nROWS WITH LGANAME, PCODE AND SOURCE ALL BLANK:", all_three_blank)


NUMERIC CHECK: Latitude
Invalid values: 0
Invalid examples: []
Minimum: -37.11146251
Maximum: -28.1685927

NUMERIC CHECK: Longitude
Invalid values: 0
Invalid examples: []
Minimum: 141.4601038
Maximum: 153.6158749

NUMERIC CHECK: Number_of_plugs
Invalid values: 0
Invalid examples: []
Minimum: 1.0
Maximum: 35.0

VALUE COUNTS: Operator
'Exploren': 301
'Tesla': 260
'Chargefox': 254
'Non-networked': 217
'PLUS ES': 150
'EVX': 111
'Evie': 84
'NRMA': 78
'JOLT': 49
'EVUp': 38
'Everty': 36
'BP': 32
'Ampol': 32
'BP Australia ': 28
'Tesla Motors ': 27
'EVE Australia': 24
'Smart Charge': 24
'Evie Networks': 20
'Porsche Smart Mobility': 19
'EVSE': 18
'Fast Cities A': 17
'EVNet': 16
'NRMA Electric': 15
'Noodoe': 13
'ChargeHub': 11
'ChargePoint': 8
'Saascharge': 7
'Viva Energy A': 7
'Non-Networked': 7
'Elanga': 7
'Charge Hub': 7
'Engie': 6
'360 EV Charge': 5
'ChargePost': 5
'CasaCharge': 5
'Chargestar': 2
'Wevolt': 2
'Porsche Destination Charging': 2
'PLUS ES Manag': 2
'Charge OS': 2
'AXCharge': 1
'V

### Numeric and Categorical Check Findings

1. All latitude, longitude, and plug-count values can be converted to finite numbers. Plug counts range from 1 to 35. Further checks are needed to confirm that plug counts are integers and coordinates represent the correct locations.
2. Operator names show formatting differences, including trailing spaces and inconsistent capitalization. Some names may refer to the same operator, but this needs to be verified before they are combined.
3. Charger ratings use mixed formats, including values with units, numbers without units, and descriptions of multiple chargers. There are also 522 entries recorded as "AC", which describes the charging type rather than its power.
4. The same 121 records have missing values in LGANAME, PCODE, and Source. These missing values will be retained and flagged during cleaning.

## 4. Data Cleaning
### 4.1 Basic Data Cleaning

This step removes leading and trailing spaces, converts blank values to `None`, and standardizes capitalization in the “Non-networked” operator label. Latitude and longitude are converted to numeric values and checked against global coordinate limits. Plug counts are checked to ensure they are positive integers. Records with missing values in all three fields—LGANAME, PCODE, and Source—are flagged for further review. All changes are made to a separate copy, leaving the original data unchanged.

In [11]:
from collections import Counter
import math

clean_records = [record.copy() for record in records]

trimmed_cells = Counter()
blank_cells = Counter()
operator_changes = Counter()

for record in clean_records:
    for column, value in record.items():
        if isinstance(value, str):
            stripped_value = value.strip()

            if stripped_value != value:
                trimmed_cells[column] += 1

            if stripped_value == "":
                record[column] = None
                blank_cells[column] += 1
            else:
                record[column] = stripped_value

    operator = record["Operator"]
    if operator and operator.casefold() == "non-networked":
        if operator != "Non-networked":
            operator_changes[(operator, "Non-networked")] += 1
        record["Operator"] = "Non-networked"

numeric_issues = []
coordinate_range_issues = []
plug_count_issues = []

for csv_row, record in enumerate(clean_records, start=2):
    for column in ["Latitude", "Longitude", "Number_of_plugs"]:
        original_value = record[column]

        try:
            numeric_value = float(original_value)
            if not math.isfinite(numeric_value):
                raise ValueError("Non-finite value")
        except (TypeError, ValueError):
            numeric_issues.append((csv_row, column, original_value))
            record[column] = None
        else:
            record[column] = numeric_value

    latitude = record["Latitude"]
    longitude = record["Longitude"]

    if latitude is not None and not -90 <= latitude <= 90:
        coordinate_range_issues.append((csv_row, "Latitude", latitude))

    if longitude is not None and not -180 <= longitude <= 180:
        coordinate_range_issues.append((csv_row, "Longitude", longitude))

    plugs = record["Number_of_plugs"]

    if plugs is not None:
        if plugs > 0 and plugs.is_integer():
            record["Number_of_plugs"] = int(plugs)
        else:
            plug_count_issues.append((csv_row, plugs))

    record["missing_region_metadata"] = all(
        record[column] is None
        for column in ["LGANAME", "PCODE", "Source"]
    )

print("Rows before cleaning:", len(records))
print("Rows after cleaning:", len(clean_records))

print("\nCELLS WITH SURROUNDING WHITESPACE REMOVED:")
print(dict(trimmed_cells))

print("\nBLANK CELLS STANDARDIZED TO NONE:")
print(dict(blank_cells))

print("\nOPERATOR LABEL CHANGES:")
print(dict(operator_changes))

print("\nNUMERIC CONVERSION ISSUES:", len(numeric_issues))
print(numeric_issues[:10])

print("\nGLOBAL COORDINATE RANGE ISSUES:", len(coordinate_range_issues))
print(coordinate_range_issues[:10])

print("\nINVALID PLUG COUNTS:", len(plug_count_issues))
print(plug_count_issues[:10])

print(
    "\nROWS WITH MISSING REGIONAL METADATA:",
    sum(record["missing_region_metadata"] for record in clean_records),
)

if numeric_issues or coordinate_range_issues or plug_count_issues:
    print("\nReview the flagged issues before exporting the cleaned data.")

Rows before cleaning: 1958
Rows after cleaning: 1958

CELLS WITH SURROUNDING WHITESPACE REMOVED:
{'Operator': 55, 'Station_address': 3, 'Station_name': 20}

BLANK CELLS STANDARDIZED TO NONE:
{'OBJECTID': 1837, 'Station_name': 1438, 'LGANAME': 121, 'PCODE': 121, 'Source': 121}

OPERATOR LABEL CHANGES:
{('Non-Networked', 'Non-networked'): 7}

NUMERIC CONVERSION ISSUES: 0
[]

GLOBAL COORDINATE RANGE ISSUES: 0
[]

INVALID PLUG COUNTS: 0
[]

ROWS WITH MISSING REGIONAL METADATA: 121


### Basic Cleaning Results
1. All 1,958 records were retained. Leading and trailing spaces were removed from 55 operator values, 3 station addresses, and 20 station names.
2. Blank values were converted to `None`. Missing values remain in OBJECTID, Station_name, LGANAME, PCODE, and Source.
3. Seven occurrences of “Non-Networked” were standardized to “Non-networked”.
4. All numeric conversions succeeded. Coordinates fall within global limits, and all plug counts are positive integers. These checks do not confirm whether coordinates match the actual station locations.
5. The 121 records with missing LGANAME, PCODE, and Source values were flagged for further review.

### 4.2 Charger Power Standardization

This step preserves the original charger ratings and creates separate fields for numeric power and format classification. Single values with an explicit kW unit are converted to numbers. Values without units, type-only labels, and combined configurations are classified separately to avoid unsupported assumptions.

In [12]:
import re
from collections import Counter

single_power_pattern = re.compile(
    r"(\d+(?:\.\d+)?)\s*kW",
    flags=re.IGNORECASE,
)

number_only_pattern = re.compile(r"\d+(?:\.\d+)?")

combined_power_pattern = re.compile(
    r"\d+\s*[x×]\s*\d+(?:\.\d+)?\s*kW"
    r"(?:\s*&\s*\d+\s*[x×]\s*\d+(?:\.\d+)?\s*kW)+",
    flags=re.IGNORECASE,
)

for record in clean_records:
    rating = record["Charger_rating"]

    # Keep the original rating and avoid guessing unknown power values.
    record["power_kw"] = None

    if rating is None:
        record["power_format"] = "missing"

    elif single_power_pattern.fullmatch(rating):
        match = single_power_pattern.fullmatch(rating)
        power = float(match.group(1))

        if power > 0:
            record["power_kw"] = power
            record["power_format"] = "single_kw"
        else:
            record["power_format"] = "invalid_power"

    elif rating.upper() in {"AC", "DC"}:
        record["power_format"] = "type_only"

    elif number_only_pattern.fullmatch(rating):
        record["power_format"] = "unit_unspecified"

    elif combined_power_pattern.fullmatch(rating):
        record["power_format"] = "combined_configuration"

    else:
        record["power_format"] = "unrecognized"

format_counts = Counter(
    record["power_format"] for record in clean_records
)

print("POWER FORMAT COUNTS:")
for category, count in sorted(format_counts.items()):
    print(f"{category}: {count}")

known_power = [
    record["power_kw"]
    for record in clean_records
    if record["power_kw"] is not None
]

print("\nRECORDS WITH SINGLE EXPLICIT KW VALUES:", len(known_power))
print("RECORDS WITHOUT A SINGLE KW VALUE:", len(clean_records) - len(known_power))

if known_power:
    print("Minimum single power (kW):", min(known_power))
    print("Maximum single power (kW):", max(known_power))

print("\nVALUES REQUIRING REVIEW:")
review_counts = Counter(
    (record["power_format"], record["Charger_rating"])
    for record in clean_records
    if record["power_format"] != "single_kw"
)

for (category, value), count in review_counts.most_common():
    print(f"{category} | {value!r}: {count}")

assert sum(format_counts.values()) == len(clean_records)
assert all(
    original["Charger_rating"] == cleaned["Charger_rating"]
    for original, cleaned in zip(records, clean_records)
    if original["Charger_rating"] is not None
    and original["Charger_rating"].strip() == original["Charger_rating"]
    and original["Charger_rating"] != ""
)

POWER FORMAT COUNTS:
combined_configuration: 99
single_kw: 1315
type_only: 522
unit_unspecified: 22

RECORDS WITH SINGLE EXPLICIT KW VALUES: 1315
RECORDS WITHOUT A SINGLE KW VALUE: 643
Minimum single power (kW): 3.0
Maximum single power (kW): 400.0

VALUES REQUIRING REVIEW:
type_only | 'AC': 522
combined_configuration | '2x350kW & 2x175kW': 85
unit_unspecified | '22': 16
combined_configuration | '2x350kW & 6x175kW': 14
unit_unspecified | '7': 5
unit_unspecified | '50': 1


### Charger Power Standardization Results

1. Converted 1,315 explicit kW values to numbers, ranging from 3 to 400 kW.
2. Classified 522 “AC” labels, 99 combined configurations, and 22 values without units separately, leaving their single numeric power values empty.
3. Retained all original ratings and classified all 1,958 records.

### 4.3 Operator Name Standardization

This step creates a standardized operator field using an explicit mapping of selected name variants. Original labels are retained for comparison. Ambiguous or possibly truncated names are flagged for review rather than automatically merged.

In [13]:
from collections import Counter

# Standardize selected names at the brand level.
operator_mapping = {
    "Tesla Motors": "Tesla",
    "Evie Networks": "Evie",
    "BP Australia": "BP",
}

review_labels = {
    "ChargeHub",
    "Charge Hub",
    "NRMA Electric",
    "Porsche Destination Charging",
    "Porsche Smart Mobility",
    "PLUS ES Manag",
    "Fast Cities A",
    "Viva Energy A",
    "Energy Austra",
    "University of",
}

changes = Counter()
review_counts = Counter()

assert len(records) == len(clean_records)

for original, record in zip(records, clean_records):
    # Preserve the exact source label.
    record["operator_raw"] = original["Operator"]

    current_name = record["Operator"]
    standardized_name = operator_mapping.get(
        current_name, current_name
    )

    record["operator_standardized"] = standardized_name
    record["operator_needs_review"] = current_name in review_labels

    if standardized_name != current_name:
        changes[(current_name, standardized_name)] += 1

    if record["operator_needs_review"]:
        review_counts[current_name] += 1

print("OPERATOR MAPPINGS APPLIED:")
for (old_name, new_name), count in sorted(changes.items()):
    print(f"{old_name} -> {new_name}: {count}")

print("\nLABELS FLAGGED FOR REVIEW:")
for name, count in review_counts.most_common():
    print(f"{name}: {count}")

print(
    "\nDistinct labels before this step:",
    len({record["Operator"] for record in clean_records}),
)
print(
    "Distinct labels after this step:",
    len({
        record["operator_standardized"]
        for record in clean_records
    }),
)
print("Records retained:", len(clean_records))

OPERATOR MAPPINGS APPLIED:
BP Australia -> BP: 28
Evie Networks -> Evie: 20
Tesla Motors -> Tesla: 27

LABELS FLAGGED FOR REVIEW:
Porsche Smart Mobility: 19
Fast Cities A: 17
NRMA Electric: 15
ChargeHub: 11
Viva Energy A: 7
Charge Hub: 7
Porsche Destination Charging: 2
PLUS ES Manag: 2
Energy Austra: 1
University of: 1

Distinct labels before this step: 49
Distinct labels after this step: 46
Records retained: 1958


### Operator Name Standardization Results

1. Standardized 75 records: 28 from “BP Australia” to “BP”, 20 from “Evie Networks” to “Evie”, and 27 from “Tesla Motors” to “Tesla”.
2. Reduced distinct operator labels from 49 to 46.
3.  Flagged 82 records for further review without merging their operator names.
4. Retained all 1,958 records and preserved the original operator labels.


**References for operator name mapping:**

- Tesla Motors → Tesla: [Tesla corporate filing](https://ir.tesla.com/_flysystem/s3/sec/000110465925087598/tm252289-4_pre14a-gen.pdf)
- Evie Networks → Evie: [Evie official website](https://evie.com.au/contact-us/)
- BP Australia → BP: [BP Australia charging announcement](https://www.bp.com/content/dam/bp/country-sites/en_au/australia/home/media/media-releases/bp-pulse-ev-charging-launches-australia.pdf)


### 4.4 Record Identification and Potential Duplicates

Check how many records have an original ID and whether any IDs are repeated. We also look for records with the same coordinates, or the same address and operator. These records are not necessarily duplicates, as one location may have different chargers. We only identify them for review and do not remove any records.

In [14]:
from collections import Counter, defaultdict

original_ids = [
    record["OBJECTID"]
    for record in clean_records
    if record["OBJECTID"] is not None
]

id_counts = Counter(original_ids)
duplicate_ids = {
    value: count
    for value, count in id_counts.items()
    if count > 1
}

print("ORIGINAL ID CHECK:")
print("Records:", len(clean_records))
print("Non-missing IDs:", len(original_ids))
print("Distinct non-missing IDs:", len(id_counts))
print("Repeated ID values:", duplicate_ids)

coordinate_groups = defaultdict(list)
address_operator_groups = defaultdict(list)

for index, record in enumerate(clean_records):
    latitude = record["Latitude"]
    longitude = record["Longitude"]

    if latitude is not None and longitude is not None:
        coordinate_groups[(latitude, longitude)].append(index)

    address = record["Station_address"]
    operator = record["operator_standardized"]

    if address and operator:
        address_key = " ".join(address.casefold().split())
        operator_key = " ".join(operator.casefold().split())

        address_operator_groups[
            (address_key, operator_key)
        ].append(index)

shared_coordinates = {
    key: indices
    for key, indices in coordinate_groups.items()
    if len(indices) > 1
}

shared_address_operator = {
    key: indices
    for key, indices in address_operator_groups.items()
    if len(indices) > 1
}

def show_candidate_groups(title, groups, limit=5):
    print(f"\n{title}")
    print("Groups:", len(groups))
    print("Records in these groups:", sum(len(v) for v in groups.values()))

    for group_number, (key, indices) in enumerate(groups.items(), start=1):
        if group_number > limit:
            break

        print(f"\nGroup {group_number}: {key}")

        for index in indices:
            record = clean_records[index]
            print({
                "csv_row": index + 2,
                "operator": record["operator_standardized"],
                "address": record["Station_address"],
                "charger_type": record["Charger_Type"],
                "rating": record["Charger_rating"],
                "plugs": record["Number_of_plugs"],
                "latitude": record["Latitude"],
                "longitude": record["Longitude"],
            })

show_candidate_groups(
    "RECORDS WITH IDENTICAL COORDINATES",
    shared_coordinates,
)

show_candidate_groups(
    "RECORDS WITH THE SAME ADDRESS AND STANDARDIZED OPERATOR",
    shared_address_operator,
)

ORIGINAL ID CHECK:
Records: 1958
Non-missing IDs: 121
Distinct non-missing IDs: 121
Repeated ID values: {}

RECORDS WITH IDENTICAL COORDINATES
Groups: 18
Records in these groups: 38

Group 1: (-34.4366153, 150.8632982)
{'csv_row': 247, 'operator': 'Tesla', 'address': '19 Princes Hwy, Figtree NSW 2525, Australia', 'charger_type': 'DC', 'rating': '175 kW', 'plugs': 6, 'latitude': -34.4366153, 'longitude': 150.8632982}
{'csv_row': 1090, 'operator': 'Tesla', 'address': '19 Princes Hwy, Figtree NSW 2525, Australia', 'charger_type': 'DC', 'rating': '2x350kW & 2x175kW', 'plugs': 6, 'latitude': -34.4366153, 'longitude': 150.8632982}
{'csv_row': 1152, 'operator': 'Non-networked', 'address': '19 Princes Hwy, Figtree NSW 2525', 'charger_type': 'AC', 'rating': 'AC', 'plugs': 2, 'latitude': -34.4366153, 'longitude': 150.8632982}

Group 2: (-33.9098594, 151.141688)
{'csv_row': 262, 'operator': 'EVE Australia', 'address': '1A Keith St Dulwich Hill NSW 2203 Australia', 'charger_type': 'AC', 'rating': 

### Record Identification and Duplicate Check Results

1. Only 121 records have an original ID. These IDs are unique, but most records have no ID. We therefore need a new record identifier for the full dataset.
2. We found 18 groups with identical coordinates, covering 38 records. We also found 18 groups with the same address and operator, covering 36 records. Some records appear in both checks.
3. Some records may be duplicates. Others may describe different chargers at the same location. We kept all records at this stage.

### 4.5 Record IDs and Review Flags

Create a record ID from the original values of each record. This ID stays the same if the row order changes, but it may change if the source values are updated. We also flag records that share coordinates or the same address and operator. These flags help the team review possible duplicates without removing any records. Each ID identifies a source record, not a physical charging site.

In [15]:
import hashlib
import json

assert len(records) == len(clean_records)

coordinate_review_indices = {
    index
    for indices in shared_coordinates.values()
    for index in indices
}

address_review_indices = {
    index
    for indices in shared_address_operator.values()
    for index in indices
}

for index, (original, cleaned) in enumerate(
    zip(records, clean_records)
):
    source_text = json.dumps(
        original,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    source_hash = hashlib.sha256(
        source_text.encode("utf-8")
    ).hexdigest()

    cleaned["record_id"] = f"ev_{source_hash}"
    cleaned["source_file"] = "ev_20251216.csv"

    cleaned["shared_coordinates"] = (
        index in coordinate_review_indices
    )
    cleaned["shared_address_operator"] = (
        index in address_review_indices
    )
    cleaned["potential_duplicate_review"] = (
        cleaned["shared_coordinates"]
        or cleaned["shared_address_operator"]
    )

record_ids = [record["record_id"] for record in clean_records]

if len(set(record_ids)) != len(record_ids):
    raise ValueError(
        "Record IDs are not unique. Review identical source records "
        "or a possible hash collision before continuing."
    )

print("Records:", len(clean_records))
print("Unique record IDs:", len(set(record_ids)))

print(
    "Records sharing coordinates:",
    sum(record["shared_coordinates"] for record in clean_records),
)

print(
    "Records sharing an address and operator:",
    sum(record["shared_address_operator"] for record in clean_records),
)

print(
    "Records flagged by either check:",
    sum(record["potential_duplicate_review"] for record in clean_records),
)

print("\nEXAMPLE RECORD IDS:")
for record in clean_records[:3]:
    print(record["record_id"])

Records: 1958
Unique record IDs: 1958
Records sharing coordinates: 38
Records sharing an address and operator: 36
Records flagged by either check: 72

EXAMPLE RECORD IDS:
ev_35fd7b517c13af2f3539c4e4bf763c10205c189b8da6e626534017d6485fb250
ev_e795ae9310df33c9b9f8840aea501aec56db120b534e345b0fc6a3d5ce041b7b
ev_92a54038955adbf16c92062ab57a4b497be32cb889fcc2b6cb5d53a12c433e0c


### Record ID and Review Flag Results

1. Created 1,958 unique record IDs, one for each source record.
2. Flagged 38 records with shared coordinates and 36 with the same address and operator.
3. Two records appear in both checks, giving 72 records for review.
4. Kept all records. These IDs identify source records, not individual charging sites.

### 4.6 Review of Missing Regional Information

Examine the 121 records with missing LGANAME, PCODE, and Source values. We check their charger types, operators, original IDs, and addresses for common patterns. Missing values are not filled unless there is reliable supporting information.

In [16]:
from collections import Counter
import re

missing_metadata_records = [
    record for record in clean_records
    if record["missing_region_metadata"]
]

print("RECORDS UNDER REVIEW:", len(missing_metadata_records))

print("\nCHARGER TYPES:")
for value, count in Counter(
    record["Charger_Type"] for record in missing_metadata_records
).most_common():
    print(f"{value}: {count}")

print("\nOPERATORS:")
for value, count in Counter(
    record["operator_standardized"]
    for record in missing_metadata_records
).most_common():
    print(f"{value}: {count}")

print("\nPOWER FORMATS:")
for value, count in Counter(
    record["power_format"] for record in missing_metadata_records
).most_common():
    print(f"{value}: {count}")

print(
    "\nRECORDS WITH AN ORIGINAL ID:",
    sum(
        record["OBJECTID"] is not None
        for record in missing_metadata_records
    ),
)

print(
    "RECORDS ALSO FLAGGED FOR DUPLICATE REVIEW:",
    sum(
        record["potential_duplicate_review"]
        for record in missing_metadata_records
    ),
)

postcode_pattern = re.compile(
    r"\bNSW[\s,]+(\d{4})\b",
    flags=re.IGNORECASE,
)

postcode_candidates = []
no_postcode_candidate = []

for record in missing_metadata_records:
    address = record["Station_address"] or ""
    matches = set(postcode_pattern.findall(address))

    if len(matches) == 1:
        postcode_candidates.append({
            "record_id": record["record_id"],
            "address": address,
            "postcode_candidate": next(iter(matches)),
        })
    else:
        no_postcode_candidate.append(record["record_id"])

print(
    "\nRECORDS WITH ONE POSTCODE CANDIDATE AFTER NSW:",
    len(postcode_candidates),
)
print(
    "RECORDS WITHOUT ONE CLEAR CANDIDATE:",
    len(no_postcode_candidate),
)

print("\nFIRST FIVE RECORDS UNDER REVIEW:")
for record in missing_metadata_records[:5]:
    print({
        "record_id": record["record_id"],
        "OBJECTID": record["OBJECTID"],
        "address": record["Station_address"],
        "operator": record["operator_standardized"],
        "charger_type": record["Charger_Type"],
        "rating": record["Charger_rating"],
    })

print("\nFIRST FIVE POSTCODE CANDIDATES:")
for candidate in postcode_candidates[:5]:
    print(candidate)

RECORDS UNDER REVIEW: 121

CHARGER TYPES:
Upcoming: 98
AC: 17
DC: 6

OPERATORS:
BP: 28
Tesla: 27
Fast Cities A: 17
NRMA Electric: 15
Non-networked: 8
Viva Energy A: 7
Exploren: 4
EVX: 3
EVE Australia: 2
PLUS ES Manag: 2
NRMA: 1
Energy Austra: 1
Evie: 1
University of: 1
Elanga: 1
PLUS ES: 1
Chargefox: 1
EVUp: 1

POWER FORMATS:
combined_configuration: 99
unit_unspecified: 22

RECORDS WITH AN ORIGINAL ID: 121
RECORDS ALSO FLAGGED FOR DUPLICATE REVIEW: 16

RECORDS WITH ONE POSTCODE CANDIDATE AFTER NSW: 117
RECORDS WITHOUT ONE CLEAR CANDIDATE: 4

FIRST FIVE RECORDS UNDER REVIEW:
{'record_id': 'ev_fc438bcdfe619faa62ac978221bd70c754ad9d12ace3fd8401d3123e857f2af1', 'OBJECTID': '11', 'address': '422 Pacific Hwy, Artarmon NSW 2064, Australia', 'operator': 'BP', 'charger_type': 'Upcoming', 'rating': '2x350kW & 2x175kW'}
{'record_id': 'ev_e6e66a462c296ae838e9d118c565e4435766dd75cbd10a3ad0b16f00fb6fb806', 'OBJECTID': '18', 'address': '129 Pacific Hwy Ourimbah NSW 2258 Australia', 'operator': 'NRMA 

In [17]:
print("ADDRESSES WITHOUT ONE CLEAR POSTCODE CANDIDATE:")

for record in missing_metadata_records:
    address = record["Station_address"] or ""
    matches = set(postcode_pattern.findall(address))

    if len(matches) != 1:
        print({
            "record_id": record["record_id"],
            "address": address,
        })

ADDRESSES WITHOUT ONE CLEAR POSTCODE CANDIDATE:
{'record_id': 'ev_4fab8b5186f0009f50c8a24fcd1882071cab4a089b2dbacb803dc7f576d07856', 'address': 'Stoney Creek Road Car Park, 497 Forest Rd, Bexley, 2207, NSW'}
{'record_id': 'ev_241744f687d20295fa62e88a95b2baf109c5ff6c5bb0d19652aa707b6a53fed0', 'address': '135 Fairfield Rd, Guildford West 2161, NSW'}
{'record_id': 'ev_473a83c89af18f936cbf018007c633a294dfbd79d5ac5ae87ceaaf75405c755d', 'address': '16 Terminus St, Castle Hill, 2154, NSW'}
{'record_id': 'ev_3194b4c6f2a86febdfb2acee95b35c86ad12f491983d8d559440cb8bedaeec90', 'address': '1A Keith St Dulwich Hill NSW'}


In [18]:
import re
from collections import Counter

postcode_after_nsw = re.compile(
    r"\bNSW[\s,]+(\d{4})\b",
    flags=re.IGNORECASE,
)

postcode_before_nsw = re.compile(
    r"\b(\d{4})[\s,]+NSW\b",
    flags=re.IGNORECASE,
)

postcode_status_counts = Counter()
postcode_conflicts = []

for record in clean_records:
    address = record["Station_address"] or ""

    candidates = set(postcode_after_nsw.findall(address))
    candidates.update(postcode_before_nsw.findall(address))

    candidate = next(iter(candidates)) if len(candidates) == 1 else None
    existing_postcode = record["PCODE"]

    record["postcode_from_address"] = candidate

    if len(candidates) > 1:
        status = "ambiguous_address"
    elif candidate is None:
        status = "no_address_candidate"
    elif existing_postcode is None:
        status = "candidate_for_missing"
    elif candidate == existing_postcode:
        status = "matches_existing"
    else:
        status = "conflicts_with_existing"
        postcode_conflicts.append({
            "record_id": record["record_id"],
            "address": address,
            "PCODE": existing_postcode,
            "postcode_from_address": candidate,
        })

    record["postcode_check"] = status
    postcode_status_counts[status] += 1

print("POSTCODE CHECK RESULTS:")
for status, count in sorted(postcode_status_counts.items()):
    print(f"{status}: {count}")

print(
    "\nMISSING PCODE RECORDS WITH AN ADDRESS CANDIDATE:",
    sum(
        record["PCODE"] is None
        and record["postcode_from_address"] is not None
        for record in clean_records
    ),
)

print("\nFIRST TEN POSTCODE CONFLICTS:")
for conflict in postcode_conflicts[:10]:
    print(conflict)

assert sum(postcode_status_counts.values()) == len(clean_records)

POSTCODE CHECK RESULTS:
candidate_for_missing: 120
conflicts_with_existing: 35
matches_existing: 932
no_address_candidate: 871

MISSING PCODE RECORDS WITH AN ADDRESS CANDIDATE: 120

FIRST TEN POSTCODE CONFLICTS:
{'record_id': 'ev_92a54038955adbf16c92062ab57a4b497be32cb889fcc2b6cb5d53a12c433e0c', 'address': '1 - 7 Ross St, Wilcannia NSW 2836, Australia', 'PCODE': '2350', 'postcode_from_address': '2836'}
{'record_id': 'ev_7cde3d4e59cd084712a54481b1dd4295c6fd2f3ab426b9fa8c291f5893d98880', 'address': '1 Little Walker St, Casino, NSW 2470, Australia', 'PCODE': '2622', 'postcode_from_address': '2470'}
{'record_id': 'ev_8907c788c92547acc6eca4a9e7787a128eea4de10d1d8f1dbd8c7582ef885e70', 'address': '10 Victoria St, Wollongong , NSW 2500', 'PCODE': 'NSW 2500', 'postcode_from_address': '2500'}
{'record_id': 'ev_776ac39be06998cf2e32e1dc28783d7254274999809038f9a91c0f02f14e1be4', 'address': '10W Apsley St, Walcha NSW 2354, Australia', 'PCODE': '2839', 'postcode_from_address': '2354'}
{'record_id': '

### Postcode Standardization

We standardize existing postcodes to four-digit text values where the format is clear. For missing postcodes, we use a single postcode candidate from the address and record its source. Different postcode values are flagged for review. The original PCODE field is kept unchanged.

In [19]:
import re
from collections import Counter

postcode_format = re.compile(
    r"(?:NSW[\s,]+)?([0-9]{4})",
    flags=re.IGNORECASE,
)

postcode_results = Counter()
remaining_conflicts = []

for record in clean_records:
    original_postcode = record["PCODE"]
    address_postcode = record["postcode_from_address"]

    normalized_postcode = None

    if original_postcode is not None:
        match = postcode_format.fullmatch(original_postcode)
        if match:
            normalized_postcode = match.group(1)

    record["postcode_standardized"] = None
    record["postcode_source"] = None
    record["postcode_needs_review"] = False

    if original_postcode is None:
        if address_postcode is not None:
            record["postcode_standardized"] = address_postcode
            record["postcode_source"] = "address"
            status = "filled_from_address"
        else:
            record["postcode_needs_review"] = True
            status = "missing_unresolved"

    elif normalized_postcode is None:
        record["postcode_needs_review"] = True
        status = "invalid_existing_format"

    elif (
        address_postcode is not None
        and normalized_postcode != address_postcode
    ):
        record["postcode_needs_review"] = True
        status = "value_conflict"

        remaining_conflicts.append({
            "record_id": record["record_id"],
            "address": record["Station_address"],
            "PCODE": original_postcode,
            "postcode_from_address": address_postcode,
        })

    else:
        record["postcode_standardized"] = normalized_postcode
        record["postcode_source"] = "PCODE"

        if normalized_postcode != original_postcode:
            status = "existing_format_standardized"
        else:
            status = "existing_retained"

    if record["postcode_check"] == "ambiguous_address":
        record["postcode_needs_review"] = True

    record["postcode_resolution"] = status
    postcode_results[status] += 1

print("POSTCODE STANDARDIZATION RESULTS:")
for status, count in sorted(postcode_results.items()):
    print(f"{status}: {count}")

print(
    "\nRECORDS REQUIRING POSTCODE REVIEW:",
    sum(record["postcode_needs_review"] for record in clean_records),
)

print("\nFIRST FIVE REMAINING CONFLICTS:")
for conflict in remaining_conflicts[:5]:
    print(conflict)

assert sum(postcode_results.values()) == len(clean_records)

POSTCODE STANDARDIZATION RESULTS:
existing_format_standardized: 10
existing_retained: 1802
filled_from_address: 120
missing_unresolved: 1
value_conflict: 25

RECORDS REQUIRING POSTCODE REVIEW: 26

FIRST FIVE REMAINING CONFLICTS:
{'record_id': 'ev_92a54038955adbf16c92062ab57a4b497be32cb889fcc2b6cb5d53a12c433e0c', 'address': '1 - 7 Ross St, Wilcannia NSW 2836, Australia', 'PCODE': '2350', 'postcode_from_address': '2836'}
{'record_id': 'ev_7cde3d4e59cd084712a54481b1dd4295c6fd2f3ab426b9fa8c291f5893d98880', 'address': '1 Little Walker St, Casino, NSW 2470, Australia', 'PCODE': '2622', 'postcode_from_address': '2470'}
{'record_id': 'ev_776ac39be06998cf2e32e1dc28783d7254274999809038f9a91c0f02f14e1be4', 'address': '10W Apsley St, Walcha NSW 2354, Australia', 'PCODE': '2839', 'postcode_from_address': '2354'}
{'record_id': 'ev_6fddf67cd79ce70f1669da6b18638c91bb7b066b4cdf86974ec620f0e920ebea', 'address': '116 Liverpool St, Scone, NSW 2337, Australia', 'PCODE': '2880', 'postcode_from_address': '23

### Postcode Standardization Results

1. Kept 1,802 existing postcodes and standardized the format of another 10.
2. Filled 120 missing values in the standardized field using postcode candidates from addresses. These values have not been independently verified.
3. Left 25 conflicting values and one unresolved missing value empty in the standardized field for further review.
4. Preserved the original PCODE values.

### 4.7 Charging Type and Status

Keep the original Charger_Type field and separate charging technology from status. AC and DC are retained as charging types. Upcoming is recorded as a planned status, while its charging type remains unknown. An AC or DC label alone does not confirm that a charger is operational.

In [22]:
from collections import Counter

for record in clean_records:
    raw_type = record["Charger_Type"]
    normalized_type = raw_type.casefold() if raw_type else ""

    if normalized_type in {"ac", "dc"}:
        record["charger_type_standardized"] = normalized_type.upper()
        record["status_from_type"] = "not_specified"
    elif normalized_type == "upcoming":
        record["charger_type_standardized"] = None
        record["status_from_type"] = "upcoming"
    else:
        record["charger_type_standardized"] = None
        record["status_from_type"] = "unknown"

print("STANDARDIZED CHARGING TYPES:")
print(Counter(
    record["charger_type_standardized"] for record in clean_records
))

print("\nSTATUS FROM THE ORIGINAL TYPE FIELD:")
print(Counter(record["status_from_type"] for record in clean_records))

STANDARDIZED CHARGING TYPES:
Counter({'AC': 1427, 'DC': 433, None: 98})

STATUS FROM THE ORIGINAL TYPE FIELD:
Counter({'not_specified': 1860, 'upcoming': 98})


### 5. Final Validation and Export

Check the record count, unique IDs, numeric values, and field consistency before saving the cleaned dataset. Existing review flags are retained. The output contains source records, not a verified list of unique charging sites.

In [23]:
import csv
import math

# Check record count and IDs.
assert clean_records, "The cleaned dataset is empty."
assert len(clean_records) == len(records), "Unexpected record count change."

record_ids = [record["record_id"] for record in clean_records]
assert all(record_ids), "Missing record IDs."
assert len(record_ids) == len(set(record_ids)), "Duplicate record IDs."

# Check field consistency.
fieldnames = list(clean_records[0].keys())

assert all(
    set(record.keys()) == set(fieldnames)
    for record in clean_records
), "Inconsistent fields across records."

# Validate numeric values and standardized postcodes.
for record in clean_records:
    latitude = record["Latitude"]
    longitude = record["Longitude"]
    plugs = record["Number_of_plugs"]

    assert isinstance(latitude, (int, float)) and math.isfinite(latitude)
    assert isinstance(longitude, (int, float)) and math.isfinite(longitude)
    assert -90 <= latitude <= 90
    assert -180 <= longitude <= 180
    assert isinstance(plugs, int) and not isinstance(plugs, bool) and plugs > 0

    power = record["power_kw"]
    if power is not None:
        assert math.isfinite(power) and power > 0
        assert record["power_format"] == "single_kw"

    postcode = record["postcode_standardized"]
    if postcode is not None:
        assert isinstance(postcode, str)
        assert len(postcode) == 4
        assert postcode.isascii() and postcode.isdigit()

# Check exact duplicates in the cleaned original fields.
original_columns = list(records[0].keys())
normalized_rows = [
    tuple(record[column] for column in original_columns)
    for record in clean_records
]
duplicate_count = len(normalized_rows) - len(set(normalized_rows))

# Export one CSV with record_id as the first column.
fieldnames = ["record_id"] + [
    name for name in fieldnames if name != "record_id"
]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / "chargers_clean.csv"

with output_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(clean_records)

# Reopen the exported file and verify its contents.
with output_path.open(encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames == fieldnames
    exported_records = list(reader)

assert len(exported_records) == len(clean_records)
assert [
    record["record_id"] for record in exported_records
] == record_ids

print("Validation passed.")
print("Saved:", output_path.name)
print("Records:", len(exported_records))
print("Unique record IDs:", len(set(record_ids)))
print("Exact duplicates after basic cleaning:", duplicate_count)

if duplicate_count:
    print("Review normalized duplicate rows before final submission.")

Validation passed.
Saved: chargers_clean.csv
Records: 1958
Unique record IDs: 1958
Exact duplicates after basic cleaning: 0
